In [2]:
import pandas as pd
import numpy as np
import sqlite3

conn = sqlite3.connect(':memory:')

# ----------------------------------------------------
# 1. 制造数据： df_clicks (故意混入各种空格、大小写、幽灵空值)
# ----------------------------------------------------
clicks_data = {
    'click_id':   [1,       2,       3,       4,       5,       6,       7,       8],
    'user_id':    [101,     101,     102,     102,     103,     104,     105,     105],
    'device':     [' iPhonE', 'iphone', 'Android', 'ANDROID', 'PC', 'iPhone', 'iPad', ' iPad '], # 👈 脏数据：前后空格、大小写恶心至极
    'coupon_val': [50,      20,      30,      np.nan,  10,      100,     40,      60]        # 👈 优惠券面额，包含没拿到券的 NaN
}
df_clicks = pd.DataFrame(clicks_data)
df_clicks.to_sql('clicks', conn, index=False, if_exists='replace')

# ----------------------------------------------------
# 2. 制造数据： df_sales (真实下单流水，蕴含逻辑悖论)
# ----------------------------------------------------
sales_data = {
    'sale_id':    [5001,    5002,    5003,    5004,    5005,    5006],
    'user_id':    [101,     101,     102,     104,     105,     106], # 👈 106号用户是自然流量转化，根本没点过广告
    'revenue':    [1000,    200,     500,     3000,    1200,    400]
}
df_sales = pd.DataFrame(sales_data)
df_sales.to_sql('sales', conn, index=False, if_exists='replace')

print("====== 🦾 第三宇宙广告数仓已就位 =======")
print("原始点击表 clicks：")
print(df_clicks)
print("\n原始销量表 sales：")
print(df_sales)



====== 🦾 第三宇宙广告数仓已就位 =======
原始点击表 clicks：
   click_id  user_id   device  coupon_val
0         1      101   iPhonE        50.0
1         2      101   iphone        20.0
2         3      102  Android        30.0
3         4      102  ANDROID         NaN
4         5      103       PC        10.0
5         6      104   iPhone       100.0
6         7      105     iPad        40.0
7         8      105    iPad         60.0

原始销量表 sales：
   sale_id  user_id  revenue
0     5001      101     1000
1     5002      101      200
2     5003      102      500
3     5004      104     3000
4     5005      105     1200
5     5006      106      400


📋 业务需求：
广告总监现在要对 “苹果生态（iPhone/iPad）” 的广告效果进行硬核复盘。他需要找出：

凡是使用苹果设备点击过广告、且通过广告产生了真实消费转化（在sales表有购买记录）的用户。计算出这些合规用户“通过广告产生的所有个人总消费金额（Total Revenue）”，并在这个“总消费大盘”里进行全球大排队，揪出前 2 名（允许并列）的优质苹果用户。

📊 输出字段要求：
最终报表必须长成这样，一个字都不能差：
user_id | apple_device_cleaned（清洗后标准的纯小写设备名，如 iphone/ipad） | total_apple_revenue（该用户在苹果设备下的总消费金额）

In [10]:
# SQL轨道
sql_query = """
WITH clean_clicks AS (
SELECT DISTINCT cl.user_id AS user_id,LOWER(TRIM(cl.device)) AS device
FROM clicks AS cl
),
apple_clicks AS(
SELECT user_id,device AS apple_device
FROM clean_clicks
WHERE device IN ('iphone','ipad')
),
user_total_revenue AS(
SELECT  ac.user_id,
        ac.apple_device,
        SUM(s.revenue) AS total_apple_revenue
FROM apple_clicks AS ac
INNER JOIN sales AS s ON ac.user_id = s.user_id
GROUP BY ac.user_id,ac.apple_device
),
ranked_revenue AS(
    SELECT  user_id,
            apple_device,
            total_apple_revenue,
            ROW_NUMBER() OVER(ORDER BY total_apple_revenue DESC,user_id ASC) AS rnk
    FROM user_total_revenue 
)
SELECT user_id, apple_device,total_apple_revenue,rnk
FROM ranked_revenue
WHERE rnk <= 3;
"""
df_sql = pd.read_sql_query(sql_query,conn)
print(df_sql)

   user_id apple_device  total_apple_revenue  rnk
0      104       iphone                 3000    1
1      101       iphone                 1200    2
2      105         ipad                 1200    3


In [32]:
# pandas轨道
import pandas as pd
import numpy as np

df_apple_clicks = (
    df_clicks
    .assign(device=lambda df:df['device'].str.strip().str.lower())
    .query("device in ['iphone','ipad']")
    .drop_duplicates(subset=['user_id','device'],ignore_index=True)
    [['user_id','device']]
    .rename(columns={'device':'apple_device'})
)
df_final = (
    df_sales
    .merge(
        df_apple_clicks,
        on='user_id',
        how='inner'
    )
    .groupby(['user_id','apple_device'])
    .agg(total_revenue=('revenue','sum'))
    .reset_index()
    .sort_values(by=['total_revenue','user_id'],ascending=[False,True])
    .assign(rnk=lambda df:df['total_revenue'].rank(method='first',ascending=False))
    .query("rnk<=2")
    [['user_id','apple_device','total_revenue']]
    .reset_index(drop=True)
)
print(df_final.to_string(index=False))

 user_id apple_device  total_revenue
     104       iphone           3000
     101       iphone           1200
